# Models, methods and the unified fine-tuning protocol

## Setup

In [1]:
import os, sys, json, glob
from pathlib import Path
from collections import Counter

IN_COLAB = "google.colab" in sys.modules
if not Path("ner").exists() and Path("../ner").exists():
    os.chdir("..")   # parts/ notebooks run from the repository root
if IN_COLAB and not Path("ner").exists():
    !git clone -b final-submission https://github.com/lofty14/bert-ner-baseline.git
    %cd bert-ner-baseline
    !pip -q install transformers==5.22.0 accelerate==1.15.0 seqeval==1.2.2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))
pd.set_option("display.precision", 2)
print("repo root:", ROOT)

repo root: /Users/chonglichen/UNSW/COMP9444/group_project


## What we built on and what is our own
*Rubric (3 marks): models / methods are judiciously chosen and appropriately applied; if building on previous work,
identify the source and clearly delineate which parts are our own work.*

| Part | Source | Our own work |
|---|---|---|
| Pre-trained encoders | Hugging Face checkpoints (below), pinned to a commit | choice of models, casing checks |
| Token classification head, Trainer | Hugging Face `transformers` | label alignment, metric, model selection, seeds |
| Metric | `seqeval` (strict IOB2) | wrapper and checks against our own entity counts |
| DAPT (masked-LM) | standard MLM objective (Gururangan et al., 2020) | PubMed corpus collection, leak check, training and evaluation |

In [2]:
import ast

def show_source(path, names):
    """Print the source of selected top-level functions/classes without importing the module."""
    text = Path(path).read_text()
    tree = ast.parse(text)
    lines = text.splitlines()
    for node in tree.body:
        if getattr(node, "name", None) in names:
            print("\n".join(lines[node.lineno - 1:node.end_lineno]))
            print()

print(Path("ner/models.py").read_text())

"""Model registry. Every checkpoint is pinned to a Hugging Face commit."""

import json

from transformers import AutoTokenizer

MODELS = {
    # `google-bert/bert-base-cased` is the canonical repo that `bert-base-cased` redirects to.
    "bert": {
        "checkpoint": "google-bert/bert-base-cased",
        "revision": "cd5ef92a9fb2f889e972770a36d4ed042daf221e",
        "lowercase": False,
    },
    # BioBERT v1.2 ships no tokenizer_config.json, so BertTokenizer would default to
    # do_lower_case=True on a cased vocab. We force case preservation.
    "biobert": {
        "checkpoint": "dmis-lab/biobert-base-cased-v1.2",
        "revision": "67c9c25b46986521ca33df05d8540da1210b3256",
        "lowercase": False,
    },
    "pubmedbert": {
        "checkpoint": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract",
        "revision": "d673b8835373c6fa116d6d8006b33d48734e305d",
        "lowercase": True,
    },
}


def load_tokenizer(key):
    spec = MODELS[key]
    tokenizer = AutoT

## Unified fine-tuning protocol
*Owner: Alex Shen.* Describe: token classification with a linear head, first-sub-word labelling, maximum length,
optimiser and learning rate, epochs, batch size, dev-set model selection on strict F1, seeds 1/2/3, fp32.

In [3]:
show_source("ner/train.py", ["encode", "to_word_level", "seqeval_scores", "compute_metrics"])

def encode(rows, tokenizer, max_length, split):
    """Tokenize words; label only the first sub-token of each word (others -100).

    Self-check: the kept labels must equal the original word labels for every sentence.
    """
    encoded = tokenizer(
        [tokens for tokens, _ in rows], truncation=True, max_length=max_length, is_split_into_words=True,
    )
    features, bad, truncated = [], [], 0
    for i, (tokens, tags) in enumerate(rows):
        word_ids = encoded.word_ids(batch_index=i)
        truncated += len(word_ids) >= max_length
        labels, previous = [], None
        for word_id in word_ids:
            if word_id is None or word_id == previous:
                labels.append(-100)
            else:
                labels.append(LABEL2ID[tags[word_id]])
            previous = word_id
        if [ID2LABEL[x] for x in labels if x != -100] != tags:
            bad.append(i)
        feature = {key: encoded[key][i] for key in encoded.keys()}
        feature["labels"] = l

In [4]:
cfg = json.load(open("dapt/results/metrics/bert_seed1.json"))
print(json.dumps({k: cfg[k] for k in ["config", "training_args", "selection"]}, indent=1)[:3000])

{
 "config": {
  "model": "bert",
  "seed": 1,
  "lr": 2e-05,
  "epochs": 3,
  "batch_size": 8,
  "eval_batch_size": null,
  "warmup_ratio": 0.0,
  "weight_decay": 0.01,
  "max_length": 512,
  "early_stopping_patience": 0,
  "save_total_limit": 2,
  "keep_checkpoints": false,
  "save_model": false,
  "train_limit": null,
  "eval_limit": null,
  "logging_steps": 50,
  "precision": "fp32",
  "select_metric": "strict_f1",
  "no_test_history": false,
  "data_dir": "data/ncbi_disease",
  "init_from": null,
  "label": "bert",
  "output_dir": "runs/v2/bert/seed1",
  "warmup_steps": 0,
  "steps_per_epoch": 679,
  "total_steps": 2037
 },
 "training_args": {
  "output_dir": "runs/v2/bert/seed1/checkpoints",
  "per_device_train_batch_size": 8,
  "num_train_epochs": 3,
  "max_steps": -1,
  "learning_rate": 2e-05,
  "lr_scheduler_type": "linear",
  "lr_scheduler_kwargs": null,
  "warmup_steps": 0,
  "optim": "adamw_torch",
  "optim_args": null,
  "weight_decay": 0.01,
  "adam_beta1": 0.9,
  "adam_b